# Suite Actuarial Mexicana -- Pipeline Completo End-to-End

Este notebook demuestra el ciclo operativo completo de una aseguradora mexicana
usando la Suite Actuarial: desde la tabla de mortalidad EMSSA-09 hasta el reporte
trimestral CNSF, pasando por tarificacion, reaseguro, reservas, RCS y validaciones SAT.

**Escenario:** Una aseguradora mediana mexicana con una cartera de seguros de vida
que necesita tarificar productos, optimizar reaseguro, estimar reservas,
calcular su RCS y generar reportes regulatorios.

In [ ]:
import sys
from datetime import date
from decimal import Decimal
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add src to path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

# Suppress matplotlib inline warnings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print(f'Suite Actuarial v{__import__("mexican_insurance").__version__}')

---
## 1. Tabla de Mortalidad EMSSA-09

La EMSSA-09 (Experiencia Mexicana del Seguro Social Actualizada 2009) es la tabla
de mortalidad oficial para seguros de vida en Mexico. Contiene probabilidades de
fallecimiento (qx) por edad y sexo.

In [ ]:
from mexican_insurance.actuarial.mortality.tablas import TablaMortalidad
from mexican_insurance.core.validators import Sexo

# Cargar la tabla oficial
tabla = TablaMortalidad.cargar_emssa09()
print(f'Tabla: {tabla.nombre}')
print(f'Registros: {len(tabla.datos)}')
tabla.datos.head(10)

In [ ]:
# Curvas de mortalidad por sexo
edades = range(18, 100)
qx_h = [float(tabla.obtener_qx(e, Sexo.HOMBRE)) for e in edades]
qx_m = [float(tabla.obtener_qx(e, Sexo.MUJER)) for e in edades]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(list(edades), qx_h, label='Hombres', color='#1f77b4', linewidth=2)
ax1.plot(list(edades), qx_m, label='Mujeres', color='#e377c2', linewidth=2)
ax1.set_xlabel('Edad')
ax1.set_ylabel('qx (probabilidad de fallecimiento)')
ax1.set_title('Curvas de Mortalidad EMSSA-09')
ax1.legend()

ax2.semilogy(list(edades), qx_h, label='Hombres', color='#1f77b4', linewidth=2)
ax2.semilogy(list(edades), qx_m, label='Mujeres', color='#e377c2', linewidth=2)
ax2.set_xlabel('Edad')
ax2.set_ylabel('qx (escala logaritmica)')
ax2.set_title('Curvas de Mortalidad (log)')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 2. Tarificacion de Productos de Vida

Tres productos fundamentales del mercado mexicano:
- **Temporal**: Riesgo puro. Paga solo si el asegurado fallece durante el plazo.
- **Ordinario (Vida Entera)**: Pago garantizado al fallecimiento. Cobertura vitalicia.
- **Dotal (Endowment)**: Paga por muerte O supervivencia. Componente de ahorro.

In [ ]:
from mexican_insurance.core.validators import Asegurado, ConfiguracionProducto, Moneda
from mexican_insurance.products.vida.temporal import VidaTemporal
from mexican_insurance.products.vida.ordinario import VidaOrdinario
from mexican_insurance.products.vida.dotal import VidaDotal

# Configuracion comun: 20 anios, tasa tecnica 5.5%
config = ConfiguracionProducto(
    nombre_producto='Vida 20 anios',
    plazo_years=20,
    tasa_interes_tecnico=Decimal('0.055'),
    recargo_gastos_admin=Decimal('0.05'),
    recargo_gastos_adq=Decimal('0.10'),
    recargo_utilidad=Decimal('0.03'),
)

# Tres perfiles de asegurado
perfiles = [
    Asegurado(edad=25, sexo=Sexo.HOMBRE, suma_asegurada=Decimal('1000000')),
    Asegurado(edad=40, sexo=Sexo.MUJER, suma_asegurada=Decimal('1000000')),
    Asegurado(edad=55, sexo=Sexo.HOMBRE, suma_asegurada=Decimal('1000000')),
]

# Crear productos
productos = {
    'Temporal': VidaTemporal(config, tabla),
    'Ordinario': VidaOrdinario(config, tabla),
    'Dotal': VidaDotal(config, tabla),
}

# Tarificar cada perfil en cada producto
resultados = []
for nombre_prod, producto in productos.items():
    for aseg in perfiles:
        try:
            r = producto.calcular_prima(aseg)
            resultados.append({
                'Producto': nombre_prod,
                'Edad': aseg.edad,
                'Sexo': aseg.sexo.value,
                'Prima Neta': float(r.prima_neta),
                'Prima Total': float(r.prima_total),
                'Gastos Admin': float(r.desglose_recargos.get('gastos_admin', 0)),
                'Gastos Adq': float(r.desglose_recargos.get('gastos_adq', 0)),
                'Utilidad': float(r.desglose_recargos.get('utilidad', 0)),
            })
        except (ValueError, Exception) as e:
            resultados.append({
                'Producto': nombre_prod, 'Edad': aseg.edad,
                'Sexo': aseg.sexo.value, 'Prima Neta': None,
                'Prima Total': None, 'Gastos Admin': None,
                'Gastos Adq': None, 'Utilidad': None,
            })

df_primas = pd.DataFrame(resultados)
df_primas.style.format({'Prima Neta': '${:,.2f}', 'Prima Total': '${:,.2f}',
                         'Gastos Admin': '${:,.2f}', 'Gastos Adq': '${:,.2f}',
                         'Utilidad': '${:,.2f}'}, na_rep='-')

In [ ]:
# Comparacion visual de primas por producto y edad
df_valid = df_primas.dropna(subset=['Prima Total'])
pivot = df_valid.pivot_table(index='Edad', columns='Producto', values='Prima Total')

ax = pivot.plot(kind='bar', figsize=(10, 5), color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax.set_ylabel('Prima Total Anual (MXN)')
ax.set_title('Comparacion de Primas por Producto y Edad\n(SA = $1,000,000 MXN, Plazo 20 anios)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='${:,.0f}', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Evolucion de reservas matematicas (asegurado de 35 anios, hombre)
aseg_35 = Asegurado(edad=35, sexo=Sexo.HOMBRE, suma_asegurada=Decimal('1000000'))
anios = list(range(0, 21))

reservas = {}
for nombre, prod in productos.items():
    reservas[nombre] = [float(prod.calcular_reserva(aseg_35, anio=a)) for a in anios]

fig, ax = plt.subplots(figsize=(10, 5))
for nombre, vals in reservas.items():
    ax.plot(anios, vals, marker='o', markersize=3, linewidth=2, label=nombre)

ax.axhline(y=1_000_000, color='gray', linestyle='--', alpha=0.5, label='Suma Asegurada')
ax.set_xlabel('Anio de la poliza')
ax.set_ylabel('Reserva Matematica (MXN)')
ax.set_title('Evolucion de Reservas Matematicas\n(Hombre 35 anios, SA $1M)')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

---
## 3. Reaseguro -- Transferencia de Riesgo

La aseguradora transfiere parte del riesgo a reaseguradoras mediante contratos:
- **Quota Share**: Cesion proporcional fija (30% de cada riesgo)
- **Excess of Loss**: Proteccion por siniestro grande (retencion $500K, limite $2M)

In [ ]:
from mexican_insurance.core.validators import (
    QuotaShareConfig, ExcessOfLossConfig, Siniestro, TipoSiniestro
)
from mexican_insurance.reinsurance.quota_share import QuotaShare
from mexican_insurance.reinsurance.excess_of_loss import ExcessOfLoss

# Contrato Quota Share: 30% cesion, 25% comision
config_qs = QuotaShareConfig(
    tipo_contrato='quota_share',
    vigencia_inicio=date(2025, 1, 1),
    vigencia_fin=date(2025, 12, 31),
    porcentaje_cesion=Decimal('30'),
    comision_reaseguro=Decimal('25'),
)
qs = QuotaShare(config_qs)

# Contrato XoL: retencion $500K, limite $2M
config_xol = ExcessOfLossConfig(
    tipo_contrato='excess_of_loss',
    vigencia_inicio=date(2025, 1, 1),
    vigencia_fin=date(2025, 12, 31),
    retencion=Decimal('500000'),
    limite=Decimal('2000000'),
    tasa_prima=Decimal('5'),
)
xol = ExcessOfLoss(config_xol)

# Siniestros de ejemplo (mezcla de chicos y grandes)
siniestros = [
    Siniestro(id_siniestro='S001', fecha_ocurrencia=date(2025, 2, 15), monto_bruto=Decimal('150000')),
    Siniestro(id_siniestro='S002', fecha_ocurrencia=date(2025, 4, 3), monto_bruto=Decimal('350000')),
    Siniestro(id_siniestro='S003', fecha_ocurrencia=date(2025, 6, 20), monto_bruto=Decimal('800000')),
    Siniestro(id_siniestro='S004', fecha_ocurrencia=date(2025, 8, 10), monto_bruto=Decimal('1500000')),
    Siniestro(id_siniestro='S005', fecha_ocurrencia=date(2025, 11, 5), monto_bruto=Decimal('200000')),
]

total_bruto = sum(s.monto_bruto for s in siniestros)
print(f'Siniestralidad bruta total: ${total_bruto:,.2f} MXN')
print(f'Numero de siniestros: {len(siniestros)}')

In [ ]:
# Aplicar Quota Share
prima_bruta_cartera = Decimal('5000000')  # $5M de primas brutas
resultado_qs = qs.calcular_resultado_neto(prima_bruta_cartera, siniestros)

print('=== QUOTA SHARE (30% cesion, 25% comision) ===')
print(f'Siniestros cedidos:    ${resultado_qs.monto_cedido:>14,.2f}')
print(f'Siniestros retenidos:  ${resultado_qs.monto_retenido:>14,.2f}')
print(f'Recuperacion reaseg:   ${resultado_qs.recuperacion_reaseguro:>14,.2f}')
print(f'Comision recibida:     ${resultado_qs.comision_recibida:>14,.2f}')
print(f'Resultado neto:        ${resultado_qs.resultado_neto_cedente:>14,.2f}')
print()

# Aplicar XoL
prima_xol = Decimal('250000')  # Prima de reaseguro pagada
resultado_xol = xol.calcular_resultado_neto(prima_xol, siniestros)

print('=== EXCESS OF LOSS (ret $500K, lim $2M) ===')
print(f'Recuperacion reaseg:   ${resultado_xol.recuperacion_reaseguro:>14,.2f}')
print(f'Retenido por cedente:  ${resultado_xol.monto_retenido:>14,.2f}')
print(f'Prima pagada:          ${resultado_xol.prima_reaseguro_pagada:>14,.2f}')
print(f'Resultado neto:        ${resultado_xol.resultado_neto_cedente:>14,.2f}')

In [ ]:
# Visualizar impacto del reaseguro por siniestro
montos = [float(s.monto_bruto) for s in siniestros]
ids = [s.id_siniestro for s in siniestros]
retencion_xol = float(config_xol.retencion)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# QS: 70% retenido, 30% cedido
retenido_qs = [m * 0.70 for m in montos]
cedido_qs = [m * 0.30 for m in montos]
ax1.bar(ids, retenido_qs, label='Retenido (70%)', color='#1f77b4')
ax1.bar(ids, cedido_qs, bottom=retenido_qs, label='Cedido (30%)', color='#ff7f0e')
ax1.set_ylabel('Monto (MXN)')
ax1.set_title('Quota Share -- Retenido vs Cedido')
ax1.legend()
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# XoL: retencion hasta $500K, exceso recuperado
retenido_xol = [min(m, retencion_xol) for m in montos]
recuperado_xol = [max(0, m - retencion_xol) for m in montos]
ax2.bar(ids, retenido_xol, label=f'Retenido (hasta ${retencion_xol:,.0f})', color='#1f77b4')
ax2.bar(ids, recuperado_xol, bottom=retenido_xol, label='Recuperacion XoL', color='#2ca02c')
ax2.axhline(y=retencion_xol, color='red', linestyle='--', alpha=0.7, label='Retencion')
ax2.set_ylabel('Monto (MXN)')
ax2.set_title('Excess of Loss -- Retenido vs Recuperado')
ax2.legend()
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

---
## 4. Reservas -- Chain Ladder, Bornhuetter-Ferguson, Bootstrap

Estimacion de reservas IBNR (Incurred But Not Reported) usando tres metodos
complementarios sobre un triangulo de desarrollo de siniestros.

In [ ]:
from mexican_insurance.core.validators import (
    ConfiguracionChainLadder, ConfiguracionBornhuetterFerguson, ConfiguracionBootstrap
)
from mexican_insurance.reservas.chain_ladder import ChainLadder
from mexican_insurance.reservas.bornhuetter_ferguson import BornhuetterFerguson
from mexican_insurance.reservas.bootstrap import Bootstrap

# Triangulo de desarrollo (pagos acumulados, miles MXN)
triangulo = pd.DataFrame({
    0: [3500, 3800, 4100, 4300, 4500, 4800],
    1: [5200, 5600, 6100, 6400, 6700, None],
    2: [6100, 6600, 7100, 7500, None, None],
    3: [6600, 7100, 7700, None, None, None],
    4: [6900, 7400, None, None, None, None],
    5: [7050, None, None, None, None, None],
}, index=[2019, 2020, 2021, 2022, 2023, 2024])

print('Triangulo de Desarrollo (pagos acumulados, miles MXN):')
triangulo

In [ ]:
# Chain Ladder
cl = ChainLadder(ConfiguracionChainLadder())
res_cl = cl.calcular(triangulo)

print('=== CHAIN LADDER ===')
print(f'Reserva IBNR total:  ${float(res_cl.reserva_total):>12,.0f}')
print(f'Ultimate total:      ${float(res_cl.ultimate_total):>12,.0f}')
print(f'Pagado total:        ${float(res_cl.pagado_total):>12,.0f}')
print()
print('Factores de desarrollo:')
for i, f in enumerate(res_cl.factores_desarrollo):
    print(f'  {i} -> {i+1}: {float(f):.4f}')
print()
print('Reservas por anio de origen:')
for anio, reserva in res_cl.reservas_por_anio.items():
    print(f'  {anio}: ${float(reserva):>10,.0f}')

In [ ]:
# Bornhuetter-Ferguson
primas = {2019: Decimal('8000'), 2020: Decimal('8500'), 2021: Decimal('9000'),
          2022: Decimal('9500'), 2023: Decimal('10000'), 2024: Decimal('10500')}

bf = BornhuetterFerguson(ConfiguracionBornhuetterFerguson(loss_ratio_apriori=Decimal('0.75')))
res_bf = bf.calcular(triangulo, primas)

print('=== BORNHUETTER-FERGUSON (LR a priori = 75%) ===')
print(f'Reserva IBNR total:  ${float(res_bf.reserva_total):>12,.0f}')
print(f'Ultimate total:      ${float(res_bf.ultimate_total):>12,.0f}')
print()
print('Reservas por anio:')
for anio, reserva in res_bf.reservas_por_anio.items():
    print(f'  {anio}: ${float(reserva):>10,.0f}')

In [ ]:
# Bootstrap (simulacion Monte Carlo)
bs = Bootstrap(ConfiguracionBootstrap(num_simulaciones=1000, seed=42, percentiles=[50, 75, 90, 95, 99]))
res_bs = bs.calcular(triangulo)

print('=== BOOTSTRAP (1,000 simulaciones) ===')
print(f'Reserva P50 (mediana): ${float(res_bs.reserva_total):>12,.0f}')
print(f'Media:                 ${float(res_bs.detalles["media"]):>12,.0f}')
print(f'Desv. Estandar:        ${float(res_bs.detalles["desviacion_estandar"]):>12,.0f}')
print(f'Coef. Variacion:       {float(res_bs.detalles["coeficiente_variacion"]):.2%}')
print()
print('Percentiles:')
for p, v in res_bs.percentiles.items():
    print(f'  P{p}: ${float(v):>12,.0f}')
print()

# VaR y TVaR
var_95 = bs.calcular_var(0.95)
tvar_95 = bs.calcular_tvar(0.95)
print(f'VaR 95%:  ${float(var_95):>12,.0f}')
print(f'TVaR 95%: ${float(tvar_95):>12,.0f}')

In [ ]:
# Comparacion de los tres metodos + distribucion Bootstrap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparativo
metodos = ['Chain Ladder', 'Bornhuetter-\nFerguson', 'Bootstrap\n(P50)']
reservas_totales = [float(res_cl.reserva_total), float(res_bf.reserva_total), float(res_bs.reserva_total)]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

bars = ax1.bar(metodos, reservas_totales, color=colors)
for bar, val in zip(bars, reservas_totales):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'${val:,.0f}', ha='center', fontsize=10)
ax1.set_ylabel('Reserva IBNR Total')
ax1.set_title('Comparacion de Metodos de Reservas')

# Histograma Bootstrap
distribucion = bs.obtener_distribucion()
if distribucion:
    dist_float = [float(d) for d in distribucion]
    ax2.hist(dist_float, bins=40, color='#2ca02c', alpha=0.7, edgecolor='white')
    ax2.axvline(float(res_bs.percentiles[50]), color='blue', linestyle='--', label=f'P50: ${float(res_bs.percentiles[50]):,.0f}')
    ax2.axvline(float(res_bs.percentiles[95]), color='red', linestyle='--', label=f'P95: ${float(res_bs.percentiles[95]):,.0f}')
    ax2.set_xlabel('Reserva IBNR Total')
    ax2.set_ylabel('Frecuencia')
    ax2.set_title('Distribucion Bootstrap (1,000 sims)')
    ax2.legend()

plt.tight_layout()
plt.show()

---
## 5. Calculo de RCS (Requerimiento de Capital de Solvencia)

El RCS es el capital minimo que una aseguradora debe mantener bajo la LISF.
Se calcula por tres tipos de riesgo (vida, danos, inversion) y se agrega
usando una matriz de correlacion definida por la CNSF.

In [ ]:
from mexican_insurance.core.validators import (
    ConfiguracionRCSVida, ConfiguracionRCSDanos, ConfiguracionRCSInversion
)
from mexican_insurance.regulatorio.agregador_rcs import AgregadorRCS

# Datos de la aseguradora (en pesos)
config_vida = ConfiguracionRCSVida(
    suma_asegurada_total=Decimal('5000000000'),    # $5,000M
    reserva_matematica=Decimal('800000000'),        # $800M
    edad_promedio_asegurados=42,
    duracion_promedio_polizas=15,
    numero_asegurados=50000,
)

config_danos = ConfiguracionRCSDanos(
    primas_retenidas_12m=Decimal('2000000000'),     # $2,000M
    reserva_siniestros=Decimal('500000000'),         # $500M
    coeficiente_variacion=Decimal('0.15'),
    numero_ramos=5,
)

config_inversion = ConfiguracionRCSInversion(
    valor_acciones=Decimal('200000000'),             # $200M
    valor_bonos_gubernamentales=Decimal('1500000000'),# $1,500M
    valor_bonos_corporativos=Decimal('500000000'),    # $500M
    valor_inmuebles=Decimal('300000000'),             # $300M
    duracion_promedio_bonos=Decimal('7.5'),
    calificacion_promedio_bonos='AA',
)

# Calcular RCS agregado
agregador = AgregadorRCS(
    config_vida=config_vida,
    config_danos=config_danos,
    config_inversion=config_inversion,
    capital_minimo_pagado=Decimal('1200000000'),  # $1,200M capital
)

resultado_rcs = agregador.calcular_rcs_completo()

print('=== RESULTADO RCS ===')
print(f'RCS Vida:       ${float(resultado_rcs.rcs_suscripcion_vida)/1e6:>10,.1f}M')
print(f'RCS Danos:      ${float(resultado_rcs.rcs_suscripcion_danos)/1e6:>10,.1f}M')
print(f'RCS Inversion:  ${float(resultado_rcs.rcs_inversion)/1e6:>10,.1f}M')
print(f'RCS Total:      ${float(resultado_rcs.rcs_total)/1e6:>10,.1f}M')
print(f'Capital:        ${float(resultado_rcs.capital_minimo_pagado)/1e6:>10,.1f}M')
print(f'Excedente:      ${float(resultado_rcs.excedente_solvencia)/1e6:>10,.1f}M')
print(f'Ratio Solv.:    {float(resultado_rcs.ratio_solvencia):.2f}')
print(f'Cumple:         {"SI" if resultado_rcs.cumple_regulacion else "NO"}')

In [ ]:
# Desglose visual del RCS
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Componentes individuales
componentes = {
    'Mortalidad': float(resultado_rcs.rcs_mortalidad)/1e6,
    'Longevidad': float(resultado_rcs.rcs_longevidad)/1e6,
    'Invalidez': float(resultado_rcs.rcs_invalidez)/1e6,
    'Gastos': float(resultado_rcs.rcs_gastos)/1e6,
    'Prima': float(resultado_rcs.rcs_prima)/1e6,
    'Reserva': float(resultado_rcs.rcs_reserva)/1e6,
    'Mercado': float(resultado_rcs.rcs_mercado)/1e6,
    'Credito': float(resultado_rcs.rcs_credito)/1e6,
}
comp_filtrado = {k: v for k, v in componentes.items() if v > 0}

bars = ax1.barh(list(comp_filtrado.keys()), list(comp_filtrado.values()), color='#1f77b4')
for bar, val in zip(bars, comp_filtrado.values()):
    ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             f'${val:,.1f}M', va='center', fontsize=9)
ax1.set_xlabel('RCS (Millones MXN)')
ax1.set_title('Desglose de RCS por Componente de Riesgo')

# Pie chart por categoria
categorias = {
    'Vida': float(resultado_rcs.rcs_suscripcion_vida)/1e6,
    'Danos': float(resultado_rcs.rcs_suscripcion_danos)/1e6,
    'Inversion': float(resultado_rcs.rcs_inversion)/1e6,
}
cat_filtrado = {k: v for k, v in categorias.items() if v > 0}
ax2.pie(cat_filtrado.values(), labels=cat_filtrado.keys(), autopct='%1.1f%%',
        colors=['#1f77b4', '#ff7f0e', '#2ca02c'], startangle=90)
ax2.set_title(f'Distribucion del RCS Total\n(${float(resultado_rcs.rcs_total)/1e6:,.1f}M)')

plt.tight_layout()
plt.show()

---
## 6. Validaciones Fiscales SAT

Validacion de deducibilidad de primas y calculo de retenciones ISR
conforme a la Ley del Impuesto Sobre la Renta.

In [ ]:
from mexican_insurance.regulatorio.validaciones_sat.models import TipoSeguroFiscal
from mexican_insurance.regulatorio.validaciones_sat.validador_primas import ValidadorPrimasDeducibles
from mexican_insurance.regulatorio.validaciones_sat.validador_retenciones import CalculadoraRetencionesISR

# Deducibilidad de primas para persona fisica
validador = ValidadorPrimasDeducibles(uma_anual=Decimal('39000'))

tipos_seguro = [
    ('Gastos Medicos', TipoSeguroFiscal.GASTOS_MEDICOS, Decimal('45000')),
    ('Vida', TipoSeguroFiscal.VIDA, Decimal('30000')),
    ('Pensiones', TipoSeguroFiscal.PENSIONES, Decimal('80000')),
    ('Danos', TipoSeguroFiscal.DANOS, Decimal('25000')),
]

print('=== DEDUCIBILIDAD DE PRIMAS (Persona Fisica) ===')
print(f'{"Tipo":<20} {"Prima":<15} {"Deducible?":<12} {"Monto Ded.":<15} {"Fundamento"}')
print('-' * 90)

for nombre, tipo, prima in tipos_seguro:
    r = validador.validar_deducibilidad(
        tipo_seguro=tipo, monto_prima=prima, es_persona_fisica=True
    )
    print(f'{nombre:<20} ${float(prima):>10,.2f}   '
          f'{"SI" if r.es_deducible else "NO":<12} '
          f'${float(r.monto_deducible):>10,.2f}   '
          f'{r.fundamento_legal[:40]}')

In [ ]:
# Retenciones ISR en pagos de siniestros
calculadora_isr = CalculadoraRetencionesISR()

pagos = [
    ('Indemnizacion muerte', TipoSeguroFiscal.VIDA, Decimal('1000000'), Decimal('0'), False, False),
    ('Gastos medicos', TipoSeguroFiscal.GASTOS_MEDICOS, Decimal('500000'), Decimal('0'), False, False),
    ('Renta vitalicia', TipoSeguroFiscal.PENSIONES, Decimal('200000'), Decimal('100000'), True, False),
    ('Retiro ahorro', TipoSeguroFiscal.VIDA, Decimal('300000'), Decimal('150000'), False, True),
]

print('=== RETENCIONES ISR EN PAGOS ===')
print(f'{"Concepto":<25} {"Monto":<15} {"Gravable":<15} {"Retencion":<15} {"Neto"}')
print('-' * 90)

for concepto, tipo, monto, gravable, es_renta, es_retiro in pagos:
    r = calculadora_isr.calcular_retencion(
        tipo_seguro=tipo, monto_pago=monto, monto_gravable=gravable,
        es_renta_vitalicia=es_renta, es_retiro_ahorro=es_retiro,
    )
    print(f'{concepto:<25} ${float(monto):>10,.0f}   '
          f'${float(gravable):>10,.0f}   '
          f'${float(r.monto_retencion):>10,.0f}   '
          f'${float(r.monto_neto_pagar):>10,.0f}')

---
## 7. Generacion de Reporte CNSF

Exportacion de resultados en formato Excel para entrega a la CNSF.

In [ ]:
from mexican_insurance.reportes.exportadores import ExportadorExcel

# Preparar DataFrames para el reporte
df_suscripcion = df_primas.dropna(subset=['Prima Total']).copy()
df_suscripcion.rename(columns={'Prima Total': 'Prima_Total_MXN'}, inplace=True)

df_rcs = pd.DataFrame([{
    'Componente': k,
    'RCS_MXN': float(v),
} for k, v in componentes.items() if float(v) > 0])

# Exportar
exportador = ExportadorExcel()
ruta = str(ROOT / 'outputs' / 'reporte_trimestral_ejemplo.xlsx')

ruta_salida = exportador.exportar_reporte_completo(
    ruta_salida=ruta,
    df_suscripcion=df_suscripcion,
    df_rcs=df_rcs,
    metadata={
        'aseguradora': 'Ejemplo S.A. de C.V.',
        'trimestre': 'Q1 2025',
        'fecha_generacion': str(date.today()),
    }
)

print(f'Reporte generado: {ruta_salida}')
print(f'Tamano: {ruta_salida.stat().st_size / 1024:.1f} KB')

---
## Resumen del Pipeline

| Fase | Modulo | Resultado |
|------|--------|-----------|
| 1. Mortalidad | `TablaMortalidad.cargar_emssa09()` | Tabla EMSSA-09 cargada |
| 2. Tarificacion | `VidaTemporal/Ordinario/Dotal.calcular_prima()` | Primas para 3 productos x 3 perfiles |
| 3. Reaseguro | `QuotaShare/ExcessOfLoss.calcular_resultado_neto()` | Retencion y recuperacion por contrato |
| 4. Reservas | `ChainLadder/BF/Bootstrap.calcular()` | IBNR con 3 metodos + intervalos de confianza |
| 5. RCS | `AgregadorRCS.calcular_rcs_completo()` | Capital requerido con desglose por riesgo |
| 6. SAT | `ValidadorPrimas/CalculadoraRetenciones` | Deducibilidad y retenciones ISR |
| 7. Reporte | `ExportadorExcel.exportar_reporte_completo()` | Excel listo para CNSF |